In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
spark = SparkSession.builder.appName("complexJson").master("local").getOrCreate()

In [3]:
df = spark.read.format("json").option("multiline", True).load("./inputData/comp.json")
df.printSchema()
df.show()


root
 |-- batters: struct (nullable = true)
 |    |-- batter: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- id: string (nullable = true)
 |    |    |    |-- type: string (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- ppu: double (nullable = true)
 |-- topping: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- type: string (nullable = true)
 |-- type: string (nullable = true)

+--------------------+----+----+----+--------------------+-----+
|             batters|  id|name| ppu|             topping| type|
+--------------------+----+----+----+--------------------+-----+
|{[{1001, Regular}...|0001|Cake|0.55|[{5001, None}, {5...|donut|
+--------------------+----+----+----+--------------------+-----+



In [4]:
df.show(truncate=False)
# df = df.withColumn("topping_explode")

+-------------------------------------------------------------------------------+----+----+----+-----------------------------------------------------------------------------------------------------------------------------------------+-----+
|batters                                                                        |id  |name|ppu |topping                                                                                                                                  |type |
+-------------------------------------------------------------------------------+----+----+----+-----------------------------------------------------------------------------------------------------------------------------------------+-----+
|{[{1001, Regular}, {1002, Chocolate}, {1003, Blueberry}, {1004, Devil's Food}]}|0001|Cake|0.55|[{5001, None}, {5002, Glazed}, {5005, Sugar}, {5007, Powdered Sugar}, {5006, Chocolate with Sprinkles}, {5003, Chocolate}, {5004, Maple}]|donut|
+-----------------------------------

In [5]:
df = df.withColumn("topping_explode", explode("topping"))
#df.select('topping', "topping_explode").show(100,truncate=False)

In [6]:
df.printSchema()

root
 |-- batters: struct (nullable = true)
 |    |-- batter: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- id: string (nullable = true)
 |    |    |    |-- type: string (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- ppu: double (nullable = true)
 |-- topping: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- type: string (nullable = true)
 |-- type: string (nullable = true)
 |-- topping_explode: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- type: string (nullable = true)



In [7]:
df = df.withColumn("topping_id", col('topping_explode.id'))\
    .withColumn("topping_type", col("topping_explode.type"))
df.show()


+--------------------+----+----+----+--------------------+-----+--------------------+----------+--------------------+
|             batters|  id|name| ppu|             topping| type|     topping_explode|topping_id|        topping_type|
+--------------------+----+----+----+--------------------+-----+--------------------+----------+--------------------+
|{[{1001, Regular}...|0001|Cake|0.55|[{5001, None}, {5...|donut|        {5001, None}|      5001|                None|
|{[{1001, Regular}...|0001|Cake|0.55|[{5001, None}, {5...|donut|      {5002, Glazed}|      5002|              Glazed|
|{[{1001, Regular}...|0001|Cake|0.55|[{5001, None}, {5...|donut|       {5005, Sugar}|      5005|               Sugar|
|{[{1001, Regular}...|0001|Cake|0.55|[{5001, None}, {5...|donut|{5007, Powdered S...|      5007|      Powdered Sugar|
|{[{1001, Regular}...|0001|Cake|0.55|[{5001, None}, {5...|donut|{5006, Chocolate ...|      5006|Chocolate with Sp...|
|{[{1001, Regular}...|0001|Cake|0.55|[{5001, None}, {5..

In [8]:
df = df.withColumn("batter_explode", explode("batters.batter"))\
        .withColumn("batter_id", col("batter_explode.id"))\
        .withColumn("batter_type", col("batter_explode.type"))
df.show(truncate=False)
df.count()
# df_final = df.drop("batters", "topping", "batter_explode","topping_explode" )
# df_final.dropDuplicates().show()

+-------------------------------------------------------------------------------+----+----+----+-----------------------------------------------------------------------------------------------------------------------------------------+-----+--------------------------------+----------+------------------------+--------------------+---------+------------+
|batters                                                                        |id  |name|ppu |topping                                                                                                                                  |type |topping_explode                 |topping_id|topping_type            |batter_explode      |batter_id|batter_type |
+-------------------------------------------------------------------------------+----+----+----+-----------------------------------------------------------------------------------------------------------------------------------------+-----+--------------------------------+----------+--------

28

In [9]:
df_final = df.drop("batters", "topping", "batter_explode","topping_explode" )

In [10]:
df_final.count()

28

In [11]:
df_final.show()

+----+----+----+-----+----------+--------------------+---------+------------+
|  id|name| ppu| type|topping_id|        topping_type|batter_id| batter_type|
+----+----+----+-----+----------+--------------------+---------+------------+
|0001|Cake|0.55|donut|      5001|                None|     1001|     Regular|
|0001|Cake|0.55|donut|      5001|                None|     1002|   Chocolate|
|0001|Cake|0.55|donut|      5001|                None|     1003|   Blueberry|
|0001|Cake|0.55|donut|      5001|                None|     1004|Devil's Food|
|0001|Cake|0.55|donut|      5002|              Glazed|     1001|     Regular|
|0001|Cake|0.55|donut|      5002|              Glazed|     1002|   Chocolate|
|0001|Cake|0.55|donut|      5002|              Glazed|     1003|   Blueberry|
|0001|Cake|0.55|donut|      5002|              Glazed|     1004|Devil's Food|
|0001|Cake|0.55|donut|      5005|               Sugar|     1001|     Regular|
|0001|Cake|0.55|donut|      5005|               Sugar|     1002|

ModuleNotFoundError: No module named 'lxml'